# Classification: Predicting Top-Price Listings

This notebook trains classifiers to label top-price listings, saves reports, and writes predictions back to BigQuery. Outputs and model settings are preserved.

This notebook keeps all modeling logic and outputs identical while adding more explanation for beginners.

### Detailed explanation
This notebook builds classification models that predict whether a listing is top-price. It covers dataset creation, preprocessing, model training, evaluation, and persistence of results.

### Functional overview
Inputs: listing data with price labels from BigQuery.
Process: preprocess features, train classifiers, evaluate them, and save artifacts.
Outputs: CSV/PNG reports and two BigQuery tables (predictions and evaluation summary).
Reason: classification teaches how to predict a category (top-price) instead of a continuous number.

**Objective:** Train classifiers to predict top-price listings.
**Inputs:** BigQuery listings with price labels.
**Process:** Build dataset, preprocess features, train models, evaluate, save artifacts.
**Outputs:** CSV/PNG reports and BigQuery tables for predictions and evaluation.
**Why:** Classification introduces categorical prediction and decision thresholds.


## 1. Imports & config

Configuration is centralized to keep paths and IDs consistent across the course.

### Configuration
We import all tools, set the random seed, and define output directories. Keeping these values centralized makes the notebook easier to follow and update.

### Why these models
Logistic regression is a clear baseline, random forest captures non-linearities, and gradient boosting offers strong performance with minimal tuning.

**Objective:** Import tools and define configuration constants.
**Inputs:** Libraries, random seed, project identifiers.
**Process:** Load modules and create report output folder.
**Outputs:** Ready environment for modeling and reporting.
**Why:** Central configuration makes the notebook easier to follow.


In [ ]:
# Objective: Import libraries, set configuration, and prepare output folders.  # Objective
# Input: Core libraries, scikit-learn tools, BigQuery client, and constants.  # Input
# Process: Import modules, define constants, and create the reports directory.  # Process
# Output: Initialized environment for classification and reporting.  # Output

from pathlib import Path  # Path handles file paths safely.
from datetime import datetime, timezone  # datetime provides timestamps and timezone utilities.

import numpy as np  # numpy supports numeric operations and reproducibility.
import pandas as pd  # pandas handles tabular data.
import matplotlib.pyplot as plt  # matplotlib creates plots.

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate  # Split and CV tools.
from sklearn.compose import ColumnTransformer  # Column-wise preprocessing.
from sklearn.pipeline import Pipeline  # Pipeline for chaining preprocessing and models.
from sklearn.impute import SimpleImputer  # Missing value imputation.
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Encoding and scaling.
from sklearn.linear_model import LogisticRegression  # Logistic regression classifier.
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier  # Tree-based models.
from sklearn.metrics import (  # Evaluation metrics for classification.
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from google.cloud import bigquery  # BigQuery client for data access and writes.



### Configuration and environment
We keep all constants and the BigQuery client setup centralized.


In [ ]:
# Configuration loaded from shared project_config.yaml


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


REPO_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(REPO_ROOT / 'config' / 'project_config.yaml')

RANDOM_SEED = int(PROJECT_CONFIG.get('random_state', 42))
project_id = str(PROJECT_CONFIG.get('gcp_project_id', 'albertheadofdata101')).strip()
dataset_id = str(PROJECT_CONFIG.get('bq_dataset', 'autoscout_audi_a3_germany')).strip()
make = str(PROJECT_CONFIG.get('make', 'audi')).strip().lower()
model = str(PROJECT_CONFIG.get('model', 'a3')).strip().lower()
country_name = str(PROJECT_CONFIG.get('country', 'germany')).strip().lower()
TAG = f'{make}_{model}_{country_name}'

REPORTS_DIR = REPO_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load external top_price label dataset from BigQuery


In [ ]:
# Objective: Load the externally labeled top_price dataset from BigQuery.  # Objective
# Input: project_id, dataset_id, and configured make/model/country scope.  # Input
# Process: Query vw_classification_dataset and keep the top_price label from price_label.  # Process
# Output: df containing features and the pre-existing binary top_price target.  # Output

client = bigquery.Client(project=project_id)


def run_query(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()


sql_dataset = f'''
SELECT
  listing_id,
  make,
  model,
  listing_country,
  actual_price_eur,
  mileage_km,
  age_years,
  power_hp,
  fuel_type,
  price_label,
  top_price
FROM `{project_id}.{dataset_id}.vw_classification_dataset`
WHERE LOWER(make) = '{make}'
  AND LOWER(model) = '{model}'
  AND LOWER(listing_country) = '{country_name}'
'''

df = run_query(sql_dataset)

if len(df) == 0:
    sql_dataset_no_country = f'''
    SELECT
      listing_id,
      make,
      model,
      listing_country,
      actual_price_eur,
      mileage_km,
      age_years,
      power_hp,
      fuel_type,
      price_label,
      top_price
    FROM `{project_id}.{dataset_id}.vw_classification_dataset`
    WHERE LOWER(make) = '{make}'
      AND LOWER(model) = '{model}'
    '''
    df = run_query(sql_dataset_no_country)
    print('No rows found for configured country; fallback scope used make/model only.')

if len(df) == 0:
    raise ValueError('No rows available for configured classification scope. Check data and config values.')

print('Loaded rows:', len(df))
print('Scope:', make, model, country_name)


## 3. Feature selection and basic validity filters

The feature list is fixed to avoid target leakage and keep results stable.

### Feature selection
We use a fixed feature list to avoid leakage and keep results consistent. We also filter out invalid values.

### Feature rationale
The features chosen are numeric and available for all listings, making them reliable inputs for a beginner classifier.

**Objective:** Select model features and filter invalid rows.
**Inputs:** DataFrame columns for price, mileage, age, fuel, and power.
**Process:** Validate columns and remove impossible values.
**Outputs:** Clean X and y ready for preprocessing.
**Why:** Input quality directly affects model performance and stability.


In [ ]:
# Objective: Select features and apply basic validity filters.  # Objective
# Input: df with listing features and external top_price target.  # Input
# Process: Define feature list, validate required columns, and filter invalid numeric values.  # Process
# Output: X (features) and y (target) filtered for modeling.  # Output

feature_cols = [
    'actual_price_eur',
    'mileage_km',
    'age_years',
    'power_hp',
    'fuel_type',
    'listing_country',
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required feature columns: {missing}')

y = df['top_price'].astype(int)
X = df[feature_cols].copy()

mask = pd.Series([True] * len(X))
if 'actual_price_eur' in X.columns:
    mask &= X['actual_price_eur'] > 0
if 'mileage_km' in X.columns:
    mask &= X['mileage_km'] > 0
if 'age_years' in X.columns:
    mask &= X['age_years'] >= 0
if 'power_hp' in X.columns:
    mask &= X['power_hp'] > 0

X = X.loc[mask].copy()
y = y.loc[mask].copy()


## 4. Preprocessing pipeline

Preprocessing is explicit so students see how numeric and categorical data are handled.

### Preprocessing
Numeric columns are imputed and scaled, and categorical columns are imputed and one-hot encoded. This creates a clean numeric matrix for modeling.

### Preprocessing rationale
We impute missing values and scale numeric features so models behave predictably. One-hot encoding makes categorical values usable by ML algorithms.

**Objective:** Prepare preprocessing for numeric and categorical data.
**Inputs:** Feature matrix with mixed types.
**Process:** Impute missing values, scale numeric features, one-hot encode categories.
**Outputs:** A reusable preprocessing transformer.
**Why:** Models require clean numeric inputs and consistent encoding.


In [ ]:
# Objective: Build preprocessing pipelines for numeric and categorical features.  # Objective
# Input: X feature matrix with mixed data types.  # Input
# Process: Define numeric and categorical pipelines and combine with ColumnTransformer.  # Process
# Output: preprocessor that can be used in model pipelines.  # Output

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()  # Identify numeric columns.
cat_cols = [c for c in X.columns if c not in numeric_cols]  # Identify categorical columns.

numeric_transformer = Pipeline(steps=[  # Pipeline for numeric features.
    ('imputer', SimpleImputer(strategy='median')),  # Fill missing numeric values.
    ('scaler', StandardScaler()),  # Standardize numeric values.
])

categorical_transformer = Pipeline(steps=[  # Pipeline for categorical features.
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Fill missing categories.
    ('onehot', OneHotEncoder(handle_unknown='ignore')),  # One-hot encode categories.
])

preprocessor = ColumnTransformer(  # Combine numeric and categorical pipelines.
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, cat_cols),
    ],
    remainder='drop',  # Drop any columns not specified.
)


## 5. Train-test split

A stratified split preserves class balance and makes metrics more reliable.

### Train-test split
A stratified split keeps the class balance similar between train and test sets.

### Evaluation approach
We use a stratified split to preserve class balance, then evaluate with accuracy, precision, recall, F1, and ROC-AUC.

**Objective:** Split the dataset for evaluation.
**Inputs:** X and y.
**Process:** Stratified train/test split with a fixed seed.
**Outputs:** Train and test sets.
**Why:** A held-out test set provides unbiased evaluation.


In [ ]:
# Objective: Split the dataset into train and test sets.  # Objective
# Input: X feature matrix and y target vector.  # Input
# Process: Perform a stratified train/test split with a fixed random seed.  # Process
# Output: X_train, X_test, y_train, y_test for modeling and evaluation.  # Output

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)  # Split data with class balance preserved.


## 6. Model definitions

We compare three standard classifiers with consistent hyperparameters.

### Models
We define three models: logistic regression, random forest, and gradient boosting. This gives students both linear and non-linear options.

**Objective:** Define three classification models.
**Inputs:** Preprocessing transformer and class balance info.
**Process:** Configure logistic regression, random forest, and gradient boosting pipelines.
**Outputs:** A dictionary of model pipelines.
**Why:** Multiple models allow comparison of linear and non-linear approaches.


In [ ]:
# Objective: Define the classification models and their settings.  # Objective
# Input: Training labels to estimate class imbalance.  # Input
# Process: Compute class ratio, set hyperparameters, create pipelines.  # Process
# Output: models dictionary containing three model pipelines.  # Output

train_counts = y_train.value_counts()  # Count class frequencies in training data.
minority_ratio = train_counts.min() / train_counts.sum()  # Compute minority class ratio.
use_class_weight = minority_ratio < 0.30  # Decide whether to apply class weights.

logreg_params = {  # Parameters for logistic regression.
    'max_iter': 2000,
    'solver': 'lbfgs',
}
if use_class_weight:  # If class imbalance is large,
    logreg_params['class_weight'] = 'balanced'  # Use balanced class weights.

model_a = Pipeline(steps=[  # Pipeline for logistic regression.
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(**logreg_params)),
])

rf_params = {  # Parameters for random forest.
    'n_estimators': 300,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
}
if use_class_weight:  # If class imbalance is large,
    rf_params['class_weight'] = 'balanced_subsample'  # Use balanced subsample weights.

model_b = Pipeline(steps=[  # Pipeline for random forest.
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(**rf_params)),
])

model_c = Pipeline(steps=[  # Pipeline for gradient boosting.
    ('preprocess', preprocessor),
    ('clf', HistGradientBoostingClassifier(random_state=RANDOM_SEED)),
])

models = {  # Dictionary of model name to pipeline.
    'LogisticRegression': model_a,
    'RandomForest': model_b,
    'HistGradientBoosting': model_c,
}


### Sanity check
Before training, we confirm class imbalance and the baseline difficulty.


## 7. Cross-validation (training only)

Cross-validation is run on training data only to avoid test leakage.

### Cross-validation
We run cross-validation only on the training set to estimate model stability without leaking test data.

### Cross-validation purpose
CV provides a stability check by evaluating models across multiple training folds.

**Objective:** Estimate model stability with cross-validation.
**Inputs:** Training data and model pipelines.
**Process:** Run stratified CV and summarize ROC-AUC and F1.
**Outputs:** CV summary table.
**Why:** CV shows how results vary across folds.


In [ ]:
# Objective: Run cross-validation on training data if possible.  # Objective
# Input: X_train, y_train, and models dictionary.  # Input
# Process: Choose folds, run CV, compute ROC-AUC and F1 summaries.  # Process
# Output: comparison_cv table with mean and std metrics.  # Output

train_counts = y_train.value_counts()  # Count class frequencies in training data.
minority_count = int(train_counts.min()) if len(train_counts) > 0 else 0  # Determine minority count.

if y_train.nunique() < 2 or minority_count < 2:  # If not enough class variety,
    comparison_cv = pd.DataFrame({  # Create a placeholder summary table.
        'model': list(models.keys()),
        'cv_roc_auc_mean': np.nan,
        'cv_roc_auc_std': np.nan,
        'cv_f1_mean': np.nan,
        'cv_f1_std': np.nan,
    })
else:  # If we have enough data for CV,
    n_splits = min(5, minority_count)  # Use up to 5 folds.
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)  # Stratified CV.

    cv_rows = []  # Collect CV results per model.
    for name, pipeline in models.items():  # Loop through each model pipeline.
        scores = cross_validate(  # Run cross-validation.
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring={'roc_auc': 'roc_auc', 'f1': 'f1'},
            n_jobs=-1,
        )
        cv_rows.append({  # Store summary stats for this model.
            'model': name,
            'cv_roc_auc_mean': scores['test_roc_auc'].mean(),
            'cv_roc_auc_std': scores['test_roc_auc'].std(),
            'cv_f1_mean': scores['test_f1'].mean(),
            'cv_f1_std': scores['test_f1'].std(),
        })

    comparison_cv = pd.DataFrame(cv_rows)  # Convert CV rows to a DataFrame.

print(comparison_cv)  # Display CV summary.


## 8. Test evaluation and comparison table

The comparison table is saved for reporting and later discussion.

### Test evaluation
We fit each model and compute accuracy, precision, recall, F1, and ROC-AUC on the test set. The combined table is saved.

**Objective:** Evaluate models on the test set and save comparison table.
**Inputs:** Train/test splits and model pipelines.
**Process:** Fit, predict, compute metrics, and save CSV.
**Outputs:** `model_comparison.csv` in reports.
**Why:** A saved comparison table supports reporting and review.


In [ ]:
# Objective: Evaluate each model on the test set and save comparison table.  # Objective
# Input: X_train, y_train, X_test, y_test, and models dictionary.  # Input
# Process: Fit models, predict, compute metrics, merge with CV summary, save CSV.  # Process
# Output: comparison table saved to reports and printed.  # Output

# Fit and evaluate on the test set  # Use test data for final evaluation.

test_rows = []  # Collect test metrics per model.
for name, pipeline in models.items():  # Loop through models.
    pipeline.fit(X_train, y_train)  # Fit model on training data.
    y_proba = pipeline.predict_proba(X_test)[:, 1]  # Predict probabilities on test data.
    y_pred = (y_proba >= 0.5).astype(int)  # Convert probabilities to class labels.

    test_rows.append({  # Store metrics for this model.
        'model': name,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred, zero_division=0),
        'test_recall': recall_score(y_test, y_pred, zero_division=0),
        'test_f1': f1_score(y_test, y_pred, zero_division=0),
        'test_roc_auc': roc_auc_score(y_test, y_proba),
    })

comparison_test = pd.DataFrame(test_rows)  # Convert test metrics to DataFrame.
comparison = comparison_test.merge(comparison_cv, on='model', how='left')  # Merge with CV summary.

comparison_path = REPORTS_DIR / f"{TAG}_model_comparison.csv"  # Define output file path.
comparison.to_csv(comparison_path, index=False)  # Save comparison table to CSV.
print('Saved comparison table to:', comparison_path)  # Confirm save location.


## 9. Confusion matrices (saved as PNG)

Confusion matrices are saved as PNGs for course slides or reports.

### Confusion matrices
Confusion matrices show true vs predicted classes and are saved as PNGs.

### Confusion matrix purpose
This shows where the model makes correct and incorrect predictions, which is easy for beginners to interpret.

**Objective:** Create confusion matrices for each model.
**Inputs:** Test labels and predictions.
**Process:** Compute confusion matrices, plot, and save PNGs.
**Outputs:** One confusion matrix image per model.
**Why:** Confusion matrices provide intuitive error breakdowns.


In [ ]:
# Objective: Plot and save confusion matrices for each model.  # Objective
# Input: X_test, y_test, and trained pipelines.  # Input
# Process: Predict labels, compute confusion matrix, plot and save PNG files.  # Process
# Output: Confusion matrix images saved to reports folder.  # Output

confusion_paths = []  # Collect saved confusion matrix paths.

for name, pipeline in models.items():  # Loop through models.
    y_proba = pipeline.predict_proba(X_test)[:, 1]  # Predict probabilities on test set.
    y_pred = (y_proba >= 0.5).astype(int)  # Convert to predicted labels.

    cm = confusion_matrix(y_test, y_pred)  # Compute confusion matrix.

    plt.figure(figsize=(4, 4))  # Create a new figure.
    plt.imshow(cm, cmap='Blues')  # Display matrix as an image.
    plt.title(f'Confusion Matrix ({TAG}) - {name}')  # Title with model name.
    plt.xlabel('Predicted')  # Label x-axis.
    plt.ylabel('Actual')  # Label y-axis.

    for i in range(cm.shape[0]):  # Loop through rows.
        for j in range(cm.shape[1]):  # Loop through columns.
            plt.text(j, i, cm[i, j], ha='center', va='center', color='black')  # Add counts.

    plt.tight_layout()  # Adjust layout.

    file_safe_name = name.replace(' ', '_')  # Create a safe filename.
    cm_path = REPORTS_DIR / f"{TAG}_confusion_matrix_{file_safe_name}.png"  # Output path.
    plt.savefig(cm_path, dpi=150)  # Save the plot.
    plt.show()  # Display the plot.

    confusion_paths.append(cm_path)  # Store the saved path.


## 10. Threshold analysis for the best model

Threshold analysis shows how precision/recall trade off for the best model.

### Threshold analysis
We evaluate how precision and recall change as we move the decision threshold for the best model.

### Threshold analysis purpose
Changing the probability threshold reveals the precision/recall trade-off, which is crucial in classification problems.

**Objective:** Analyze precision/recall trade-offs across thresholds.
**Inputs:** Best model probabilities on the test set.
**Process:** Compute metrics at multiple thresholds and save plots.
**Outputs:** Threshold CSV and precision/recall plots.
**Why:** Threshold tuning is key for classification decisions.


### Thresholding is an operating decision
We vary the probability threshold to trade precision vs recall without retraining.


In [ ]:
# Objective: Analyze precision/recall across thresholds for the best model.  # Objective
# Input: comparison table and model predictions on test data.  # Input
# Process: Select best model, compute metrics for multiple thresholds, save tables and plots.  # Process
# Output: Threshold table CSV and precision/recall plots.  # Output

# Best model by ROC-AUC, tie-break by F1  # Select best performing model.
best_row = comparison.sort_values(
    ['test_roc_auc', 'test_f1'], ascending=False
).iloc[0]  # Take top row after sorting.

best_model_name = best_row['model']  # Name of the best model.
best_model = models[best_model_name]  # Best model pipeline.

best_proba = best_model.predict_proba(X_test)[:, 1]  # Probabilities for best model.


In [ ]:
thresholds = np.arange(0.2, 0.81, 0.05)  # Thresholds to evaluate.

rows = []  # Collect threshold metrics.
for t in thresholds:  # Loop over each threshold.
    preds = (best_proba >= t).astype(int)  # Convert probabilities to labels.
    rows.append({  # Store metrics for this threshold.
        'threshold': t,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'accuracy': accuracy_score(y_test, preds),
    })

threshold_df = pd.DataFrame(rows)  # Convert metrics to DataFrame.



In [ ]:
threshold_path = REPORTS_DIR / f"{TAG}_threshold_table_{best_model_name.replace(' ', '_')}.csv"  # Output path.
threshold_df.to_csv(threshold_path, index=False)  # Save threshold table.
print('Saved threshold table to:', threshold_path)  # Confirm save.

plt.figure(figsize=(6, 4))  # Create figure for precision plot.
plt.plot(threshold_df['threshold'], threshold_df['precision'], marker='o')  # Plot precision.
plt.title(f'Precision vs Threshold ({TAG}) - {best_model_name}')  # Title.
plt.xlabel('Threshold')  # X-axis label.
plt.ylabel('Precision')  # Y-axis label.
plt.grid(True, alpha=0.3)  # Light grid for readability.
plt.tight_layout()  # Adjust layout.

precision_plot_path = REPORTS_DIR / f"{TAG}_precision_threshold_{best_model_name.replace(' ', '_')}.png"  # Output path.
plt.savefig(precision_plot_path, dpi=150)  # Save precision plot.
plt.show()  # Display precision plot.

plt.figure(figsize=(6, 4))  # Create figure for recall plot.
plt.plot(threshold_df['threshold'], threshold_df['recall'], marker='o')  # Plot recall.
plt.title(f'Recall vs Threshold ({TAG}) - {best_model_name}')  # Title.
plt.xlabel('Threshold')  # X-axis label.
plt.ylabel('Recall')  # Y-axis label.
plt.grid(True, alpha=0.3)  # Light grid for readability.
plt.tight_layout()  # Adjust layout.

recall_plot_path = REPORTS_DIR / f"{TAG}_recall_threshold_{best_model_name.replace(' ', '_')}.png"  # Output path.
plt.savefig(recall_plot_path, dpi=150)  # Save recall plot.
plt.show()  # Display recall plot.


## 11. Feature importance artifacts

Interpretability outputs are saved for classroom explanation.

### Feature interpretation
We save logistic regression coefficients and random forest importances for interpretability.

**Objective:** Save feature interpretation artifacts.
**Inputs:** Logistic regression and random forest models.
**Process:** Extract coefficients and importances, save to CSV.
**Outputs:** Two CSV files for interpretability.
**Why:** Interpretation helps students understand model behavior.


In [ ]:
# Objective: Save feature importance artifacts for interpretability.  # Objective
# Input: Trained logistic regression and random forest models.  # Input
# Process: Extract feature names, coefficients, and importances, save as CSV.  # Process
# Output: Coefficient and importance CSV files in reports.  # Output

# Logistic Regression coefficients  # Interpret linear model.
logreg_pipe = models['LogisticRegression']  # Access logistic regression pipeline.
preprocessor_fitted = logreg_pipe.named_steps['preprocess']  # Access fitted preprocessor.
clf = logreg_pipe.named_steps['clf']  # Access logistic regression classifier.

feature_names = preprocessor_fitted.get_feature_names_out()  # Get feature names after encoding.
coefs = clf.coef_.ravel()  # Get coefficient values as a flat array.

coef_df = pd.DataFrame({  # Build coefficients table.
    'feature': feature_names,
    'coefficient': coefs,
}).sort_values('coefficient', ascending=False)  # Sort by coefficient.

coef_path = REPORTS_DIR / f"{TAG}_logreg_top_coefficients.csv"  # Output path for coefficients.
coef_df.to_csv(coef_path, index=False)  # Save coefficients CSV.
print('Saved coefficients to:', coef_path)  # Confirm save.

# Random Forest feature importances  # Interpret tree-based model.
rf_pipe = models['RandomForest']  # Access random forest pipeline.
rf_clf = rf_pipe.named_steps['clf']  # Access random forest classifier.

rf_feature_names = rf_pipe.named_steps['preprocess'].get_feature_names_out()  # Feature names after encoding.
rf_importances = rf_clf.feature_importances_  # Importance scores from random forest.

rf_imp_df = pd.DataFrame({  # Build importances table.
    'feature': rf_feature_names,
    'importance': rf_importances,
}).sort_values('importance', ascending=False)  # Sort by importance.

rf_imp_path = REPORTS_DIR / f"{TAG}_rf_feature_importance.csv"  # Output path for importances.
rf_imp_df.to_csv(rf_imp_path, index=False)  # Save importances CSV.
print('Saved RF importances to:', rf_imp_path)  # Confirm save.


### Persistence contract (schema must remain stable)
This step writes per-listing predictions with a fixed schema used by BI.


## 12. Persist predictions to BigQuery

Predictions are written to BigQuery with the required schema.

### BigQuery persistence
We write prediction probabilities and evaluation summaries back to BigQuery with fixed table names and schemas.

### Persistence purpose
Saving predictions and evaluation summaries ensures results can be used in dashboards or further analysis.

**Objective:** Write per-listing predictions to BigQuery.
**Inputs:** Full dataset and trained models.
**Process:** Predict probabilities, assemble schema, overwrite table.
**Outputs:** BigQuery table `fact_top_price_predictions`.
**Why:** Persisted predictions support downstream analytics.


In [ ]:
# Objective: Persist per-listing predictions to BigQuery.  # Objective
# Input: Full feature set X and trained model pipelines.  # Input
# Process: Predict probabilities for each model, build combined table, write to BigQuery.  # Process
# Output: fact_top_price_predictions table in BigQuery.  # Output

model_name_map = {  # Map pipeline names to schema values.
    'LogisticRegression': 'logistic_regression',
    'RandomForest': 'random_forest',
    'HistGradientBoosting': 'hist_gradient_boosting',
}

# Align listing_id with the filtered feature set  # Keep IDs consistent with X.
# We use df filtered by X.index to stay consistent with preprocessing.  # Preserve alignment.
df_modeling = df.loc[X.index].copy()  # Subset df to the same rows as X.
if 'listing_id' not in df_modeling.columns:  # Validate listing_id presence.
    raise ValueError('listing_id is required for BigQuery persistence.')  # Stop if missing.

threshold_used = 0.5  # Fixed threshold used for predicted_label.

pred_rows = []  # Collect per-model prediction tables.
for name, pipeline in models.items():  # Loop through models.
    proba = pipeline.predict_proba(X)[:, 1]  # Predict probabilities for all rows.
    model_df = pd.DataFrame({  # Build prediction rows for this model.
        'listing_id': df_modeling['listing_id'].astype('int64'),
        'model_name': model_name_map[name],
        'predicted_proba': proba.astype('float64'),
        'predicted_label': (proba >= threshold_used),
        'threshold_used': float(threshold_used),
    })
    pred_rows.append(model_df)  # Append to list.

predictions_df = pd.concat(pred_rows, ignore_index=True)  # Combine all model predictions.

pred_table = f"{project_id}.{dataset_id}.fact_top_price_predictions"  # Target table name.
client.delete_table(pred_table, not_found_ok=True)  # Delete existing table if present.
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')  # Overwrite on write.

client.load_table_from_dataframe(
    predictions_df,
    pred_table,
    job_config=job_config,
).result()  # Write predictions to BigQuery.

print('Rows written:', len(predictions_df))  # Confirm row count.
print('Table replaced:', pred_table)  # Confirm table name.


### Model governance artifact
We persist evaluation metrics so BI can compare models over time.


## 13. Persist evaluation summary to BigQuery

The evaluation summary table feeds dashboards and downstream analysis.

**Objective:** Write evaluation summary to BigQuery.
**Inputs:** Comparison metrics table.
**Process:** Rename fields, order columns, overwrite summary table.
**Outputs:** BigQuery table `model_evaluation_summary`.
**Why:** Centralized evaluation metrics are useful for reporting.


In [ ]:
# Objective: Persist evaluation summary metrics to BigQuery.  # Objective
# Input: comparison table with test and CV metrics.  # Input
# Process: Rename columns, add threshold, order schema, write to BigQuery.  # Process
# Output: model_evaluation_summary table in BigQuery.  # Output

summary_df = comparison.copy()  # Copy the comparison table.
summary_df = summary_df.rename(columns={  # Rename columns to match schema.
    'model': 'model_name',
    'test_roc_auc': 'roc_auc_test',
    'test_f1': 'f1_test',
    'test_precision': 'precision_test',
    'test_recall': 'recall_test',
    'test_accuracy': 'accuracy_test',
})

summary_df['model_name'] = summary_df['model_name'].map(model_name_map)  # Map model names.
summary_df['threshold_used'] = float(threshold_used)  # Add threshold used.

summary_df = summary_df[[  # Order columns to match expected schema.
    'model_name',
    'roc_auc_test',
    'f1_test',
    'precision_test',
    'recall_test',
    'accuracy_test',
    'cv_roc_auc_mean',
    'cv_roc_auc_std',
    'cv_f1_mean',
    'cv_f1_std',
    'threshold_used',
]]

eval_table = f"{project_id}.{dataset_id}.model_evaluation_summary"  # Target summary table.
client.delete_table(eval_table, not_found_ok=True)  # Delete existing table if present.
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')  # Overwrite on write.

client.load_table_from_dataframe(
    summary_df,
    eval_table,
    job_config=job_config,
).result()  # Write summary to BigQuery.

print('Rows written:', len(summary_df))  # Confirm row count.
print('Table replaced:', eval_table)  # Confirm table name.
